# Pelajaran 18: Memperkasakan Ejen AI dengan Resit Kriptografi

## Nota Praktikal

Nota ini membimbing melalui empat latihan:

1. **Tandatangani resit pertama anda** untuk panggilan alat ejen dan sahkan ia.
2. **Cuba ubah resit** dan lihat pengesahan gagal.
3. **Bina rantaian tiga resit** dan sahkan integriti rantaian.
4. **Balut panggilan alat Microsoft Agent Framework** supaya setiap tindakan mengeluarkan resit.

Semua primitif kriptografi diimport dari perpustakaan yang dijaga rapi (`pynacl` untuk Ed25519, `jcs` untuk JSON kanonik RFC 8785, `hashlib` dari perpustakaan standard Python untuk SHA-256). Logik resit itu sendiri adalah Python biasa yang anda boleh baca dan ubah suai.

Jalankan sel mengikut urutan. Setiap bahagian adalah pendek dan berdiri sendiri.


## Setup

Pasang dua kebergantungan tersebut. Kedua-duanya mempunyai lesen yang membenarkan (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utiliti Pembantu

Dua pembantu ini mengendalikan pengekodan base64url (tanpa padding) dan penghashan SHA-256 bagi objek sewenang-wenangnya. Mereka memastikan baki buku nota tetap tertumpu pada logik resit itu sendiri.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Tandatangani resit pertama anda

Bayangkan ejen kami untuk **Contoso Travel** baru sahaja mencari penerbangan dari Sydney ke Los Angeles untuk seorang pelanggan. Kami ingin merekod panggilan alat ini sebagai resit bertandatangan supaya juruaudit masa depan dapat mengesahkannya tanpa mempercayai kami.

### Langkah 1.1: Hasilkan kunci tandatangan

Dalam pengeluaran, kunci tandatangan ejen akan disimpan dalam modul keselamatan perkakasan (HSM), Azure Key Vault, atau stor terlindung yang serupa. Untuk pelajaran ini kami menghasilkan kunci baru dalam memori.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Langkah 1.2: Bangunkan payload resit

Payload mengandungi segala yang kita mahu resit sahkan: siapa bertindak, alat apa, dengan hujah apa, apa yang kembali, di bawah polisi apa, dan bila. Kami melakukan hash pada hujah dan hasilnya daripada memasukkannya secara terus supaya resit tidak mendedahkan kandungan sensitif.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Langkah 1.3: Tandatangan dan pasangkan resit

Tiga langkah:

1. Kanonisasikan beban menggunakan JCS supaya dua implementasi yang menghasilkan resit logik yang sama menghasilkan bait yang serupa.
2. Hash bait kanonik dengan SHA-256.
3. Tandatangani hash menggunakan kunci peribadi Ed25519.

Tandatangan tersebut kemudian dipasangkan ke beban asal untuk menghasilkan resit akhir.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Langkah 1.4: Sahkan resit

Pengesahan menyalurkan proses. Kami menanggalkan tandatangan, mengira semula hash kanonik, dan menyemak tandatangan berbanding kunci awam dalam resit.

Seorang juruaudit yang melakukan pengesahan ini tidak memerlukan apa-apa daripada kami kecuali resit itu sendiri. Tiada perkhidmatan untuk dipanggil, tiada direktori kunci untuk ditanya, tiada kepercayaan diperlukan.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Anda sepatutnya melihat `Receipt is valid: True`. Ejen telah menghasilkan rekod audit yang ditandatangani secara kriptografi yang pertama.


## Seksyen 2: Campur tangan dengan resit

Keseluruhan tujuan resit adalah supaya ia boleh dikesan jika dicemari. Mari buktikannya.

Kami akan mengubah tepat satu aksara pada resit dan melihat pengesahan gagal.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Apa yang baru sahaja berlaku?

Apabila kami menukar `policy_id`, bait kanonik berubah. Hash SHA-256 bagi bait tersebut berubah. Tandatangan (yang dibuat berdasarkan hash asal) tidak lagi sepadan dengan hash baru. Pengesahan dengan betul mengembalikan `False`.

Tiada cara untuk mengubah mana-mana medan pada resit dan masih boleh mengesahkannya, melainkan penyerang mempunyai kunci peribadi. Selagi kunci peribadi disimpan dalam peti kunci dan kunci awam diterbitkan, pengubahan tidak mungkin disembunyikan.

Cuba sendiri: ubah `tool_name` atau `agent_id` atau `timestamp` dalam sel di atas dan jalankan semula. Setiap perubahan menghasilkan resit yang tidak sah.


## Seksyen 3: Rantai resit bersama-sama

Satu resit melindungi satu tindakan. Kebanyakan agen melakukan banyak tindakan. Untuk menjadikan keseluruhan urutan itu jelas dari sebarang pengubahan, kami pautkan setiap resit kepada resit sebelumnya dengan memasukkan hash resit sebelumnya dalam beban baru resit tersebut.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jika sesiapa mengalih keluar atau menyusun semula resit, rantai akan terputus pada titik itu juga. Pengesahan mana-mana resit kemudian gagal kerana `previous_receipt_hash` tidak lagi sepadan dengan hash sebenar pendahulunya.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Sekarang putuskan rantaian dengan memanipulasi resit tengah dan sahkan semula. Resit yang dimanipulasi gagal dalam pemeriksaan tandatangannya, DAN resit seterusnya gagal dalam pemeriksaan pautan rantaian (kerana `previous_receipt_hash`nya tidak lagi sepadan dengan hash resit tengah yang diubah).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Resit 0 masih mengesahkan (ia tidak diubah dan tiada pendahulu untuk bergantung). Resit 1 gagal pemeriksaan tandatangan kerana kami mengubah `tool_args_hash`. Resit 2 gagal pemeriksaan pautan-rantai kerana `previous_receipt_hash`nya dikira terhadap resit 1 asal (yang kini diubah).

Walaupun penyerang menandatangani semula resit 1 yang diubah (yang tidak boleh mereka lakukan tanpa kunci peribadi), ketidaksesuaian pautan-rantai dalam resit 2 masih akan mendedahkan pengubahsuaian tersebut. Untuk menyembunyikan perubahan itu, penyerang perlu menandatangani semula setiap resit bermula dari titik pengubahsuaian, yang memerlukan pemilikan kunci peribadi.


## Seksyen 4: Balut panggilan alat agen dengan tandatangan resit

Dalam pengeluaran sebenar, anda tidak mahu setiap penulis agen teringat untuk memanggil `make_receipt`. Anda mahu tandatangan resit menjadi automatik untuk setiap panggilan alat.

Berikut adalah corak termudah: kelas pembungkus yang mengambil mana-mana fungsi alat boleh-panggil dan mengembalikan versi yang mengeluarkan resit. Ini boleh disesuaikan dengan mana-mana rangka kerja agen, termasuk Rangka Kerja Agen Microsoft (`agent_framework.azure`).

Jika anda tidak mempunyai projek Azure AI Foundry yang disediakan, tiruan tempatan di bawah masih menunjukkan corak tersebut.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Mengintegrasikan dengan Rangka Kerja Ejen Microsoft

Pembungkus `ReceiptedTool` di atas tidak bergantung pada rangka kerja tertentu. Untuk menggunakannya di dalam ejen yang dibina dengan Rangka Kerja Ejen Microsoft, daftar fungsi yang dibungkus sebagai alat. Satu lakaran (anda akan menggantikan mok dengan pendaftaran alat Azure AI Foundry yang sebenar):

```python
# Pseudokod yang menunjukkan bentuk integrasi.
# dari agent_framework.azure import AzureAIProjectAgentProvider
# dari azure.identity import AzureCliCredential
#
# penyedia = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# ejen = penyedia.create_agent(
#     arahan="Anda adalah ejen Pelancongan Contoso ...",
#     alat=[receipted_lookup],   # alat yang dibungkus, bukan fungsi mentah
# )
# respons = ejen.run("Cari penerbangan dari Sydney ke Los Angeles pada bulan Jun.")
#
# # Selepas run, setiap panggilan alat yang dibuat oleh ejen mempunyai resit yang ditandatangani:
# rantai_audit = receipted_lookup.receipts
```

Rangka kerja ejen tidak perlu mengetahui apa-apa tentang resit. Tandatangan resit dibungkus di sekeliling alat, bukan dipasang ke dalam rangka kerja. Inilah cara anda menambah asal-usul kepada kod ejen sedia ada tanpa menulis semula ejen tersebut.


## Rumusan dan cabaran lanjutan

Anda telah:

- Menjana pasangan kunci Ed25519.
- Membina dan menandatangani resit untuk panggilan alat ejen.
- Mengesahkan resit secara luar talian menggunakan hanya kunci awam.
- Memanipulasi resit dan melihat pengesahan gagal.
- Membina urutan rantaian hash tiga resit.
- Memanipulasi tengah rantaian dan melihat kedua-dua kegagalan tandatangan dan kegagalan pautan rantaian.
- Membungkus fungsi alat dengan penandatangan resit automatik.

**Cabaran lanjutan.** Perluaskan skema resit dengan medan `request_id` (UUID untuk penjejakan diedarkan). Kemaskini `make_receipt` untuk memasukkannya, dan sahkan resit masih boleh disahkan dari hujung ke hujung. Kemudian ubah medan itu selepas penandatanganan dan sahkan pengesahan gagal. Ini memaksa anda memahami bagaimana setiap bait pengekodan kanonik menyumbang kepada tandatangan.

**Sempadan penting.** Resit membuktikan tiga perkara sahaja: atribusi (kunci ini menandatangani kandungan ini), integriti (kandungan tidak berubah sejak penandatanganan), dan susunan (resit ini datang selepas resit itu). Mereka TIDAK membuktikan tindakan ejen adalah betul, bahawa polisi yang dinamakan dalam `policy_id` benar-benar dinilai, atau bahawa ejen mengikuti setiap peraturan. Resit adalah asas. Tadbir urus adalah sistem yang anda bina di atasnya.

Baca semula README pelajaran dengan sempadan itu dalam fikiran. Kesilapan paling biasa yang dilakukan pasukan dengan resit adalah menganggap "kami ada resit" bermakna "kami ditadbir." Tidak begitu. Resit menjadikan tingkah laku ejen boleh diaudit. Mereka tidak menjadikannya betul.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
